# 00 · Setup data PDH/PDL di Google Colab

Notebook ini menyiapkan data berlabel untuk eksperimen model (Gradient Boosting vs LSTM):

1. clone repo `xauusd-pdhl-ml` (data mentah `XAUUSD_H1.csv` sudah ada di dalamnya),
2. cek SHA-256 data mentah,
3. jalankan pipeline `configs/tesis.json` (data mentah → event → split + kunci uji → label), lalu verifikasi
   terhadap `references/tesis.json` (harus **19/19 OK**),
4. jalankan tes,
5. muat data berlabel untuk pemodelan lewat `load_model_csv`.

Runtime CPU sudah cukup. Seluruh pipeline butuh sekitar 3 detik.

In [ ]:
import os
import subprocess
import sys

REPO_URL = "https://github.com/rizalpriyono/xauusd-pdhl-ml.git"
BASE_DIR = "/content" if os.path.isdir("/content") else os.getcwd()   # Colab: /content
REPO_DIR = os.path.join(BASE_DIR, "xauusd-pdhl-ml")

if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
if os.path.join(REPO_DIR, "src") not in sys.path:
    sys.path.insert(0, os.path.join(REPO_DIR, "src"))   # paket pdhl_data hanya memakai stdlib

commit = subprocess.run(["git", "log", "-1", "--format=%h %s"], capture_output=True, text=True).stdout
print("Repo:", REPO_DIR)
print("Commit:", commit.strip())
print("Python:", sys.version.split()[0])

## Cek data mentah

SHA-256 `data/raw/XAUUSD_H1.csv` harus sama dengan acuan. Kalau berbeda, hasil pipeline tidak akan sama
dengan keluaran yang sudah diverifikasi.

In [ ]:
import json

from pdhl_data import common

reference = json.load(open("references/tesis.json"))
raw_sha = common.sha256_file("data/raw/XAUUSD_H1.csv")
assert raw_sha == reference["raw"]["sha256"], f"SHA-256 data mentah berbeda: {raw_sha}"
print("Data mentah OK:", raw_sha[:16], "…")

## Jalankan pipeline dan verifikasi

Keluaran ditulis ke `runs/tesis/`. Runtime Colab bersifat sementara, jadi jalankan ulang sel ini di setiap
sesi baru. Hasilnya deterministik dan diverifikasi otomatis.

Catatan kunci uji: Colab berjalan sebagai *root*, sehingga izin hanya-baca pada `runs/tesis/kunci_uji/`
tidak mengikat. Kunci tetap dijaga oleh manifest SHA-256 (`LockViolation` bila isinya berubah) dan oleh guard
`load_model_csv`.

In [ ]:
from pdhl_data.__main__ import main

rc = main(["run", "--config", "configs/tesis.json"])
assert rc == 0, "Verifikasi terhadap references/tesis.json gagal"

## Jalankan tes

In [ ]:
result = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests"],
                        capture_output=True, text=True)
print("\n".join(result.stderr.strip().splitlines()[-3:]))
assert result.returncode == 0, result.stderr[-3000:]

## Muat data berlabel untuk pemodelan

Kode pemodelan **wajib** memuat data lewat `load_model_csv`. Loader ini menolak folder `kunci_uji*` dan
baris dengan `session_date` > 2024-12-31.

| Berkas | Isi |
|---|---|
| `runs/tesis/data/labels_{dev,calib}.csv` | label (Reversal / Continuation / Netral) + R per event |
| `runs/tesis/data/events_{dev,calib}.csv` | event valid, urutan baris sama dengan berkas label |
| `runs/tesis/data/bars_devcalib.csv` | bar H1 s.d. 2024-12-31 (untuk urutan input LSTM, `bar_idx` ≤ `decision_bar_idx`) |

In [ ]:
import pandas as pd

from pdhl_data import TestDataLeak, load_model_csv

DATA = "runs/tesis/data"
labels_dev = pd.DataFrame(load_model_csv(f"{DATA}/labels_dev.csv"))
labels_calib = pd.DataFrame(load_model_csv(f"{DATA}/labels_calib.csv"))
events_dev = pd.DataFrame(load_model_csv(f"{DATA}/events_dev.csv"))
bars = pd.DataFrame(load_model_csv(f"{DATA}/bars_devcalib.csv"))

assert (labels_dev["event_id"].values == events_dev["event_id"].values).all()
print(f"dev: {len(labels_dev)} event, calib: {len(labels_calib)} event, bar: {len(bars)}")
display(pd.concat({"dev": labels_dev["label"].value_counts(),
                   "calib": labels_calib["label"].value_counts()}, axis=1))

try:
    load_model_csv("runs/tesis/kunci_uji/events_test.csv")
except TestDataLeak as exc:
    print("Guard bekerja, data uji ditolak:", exc)

## Catatan untuk notebook berikutnya

- **Logika di `src/`, notebook hanya memanggil.** Fitur, model, kalibrasi, dan evaluasi ditulis sebagai
  modul dan diberi tes, supaya bisa ditinjau dan dibekukan sebelum Kunci-2.
- **Simpan hasil model ke Google Drive** (`from google.colab import drive; drive.mount("/content/drive")`).
  Isi `/content` hilang saat runtime berhenti.
- **Catat versi library** (`lightgbm`, `scikit-learn`, `torch`, `numpy`) di setiap keluaran, karena Colab
  memperbarui paket tanpa pemberitahuan.
- Rencana berikutnya: `01_gradient_boosting.ipynb` dan `02_lstm.ipynb`, dengan walk-forward 2019–2023,
  *temperature scaling* di 2024, dan perbandingan *log loss* terhadap B0–B2.